---

**Load essential libraries**

---

In [2]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
%matplotlib inline

import tensorflow as tf

---

**Check TensorFlow version**

---

In [3]:
tf.__version__

'2.14.0'

---

Load MNIST Data

---

In [4]:
## Load MNIST data
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1]*X_train.shape[2])
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1]*X_test.shape[2])

num_labels = len(np.unique(y_train))
num_features = X_train.shape[1]
num_samples = X_train.shape[0]

# One-hot encode class labels
Y_train = tf.keras.utils.to_categorical(y_train)
Y_test = tf.keras.utils.to_categorical(y_test)

# Normalize the samples (images)
xmax = np.amax(X_train)
xmin = np.amin(X_train)
X_train = (X_train - xmin) / (xmax - xmin) # all train features turn into a number between 0 and 1
X_test = (X_test - xmin)/(xmax - xmin)

print('MNIST set')
print('---------------------')
print('Number of training samples = %d'%(num_samples))
print('Number of features = %d'%(num_features))
print('Number of output labels = %d'%(num_labels))

MNIST set
---------------------
Number of training samples = 60000
Number of features = 784
Number of output labels = 10


---

We will now look at 3 different ways to build custom models using TensorFlow 2:

1. model subclassing ([Making new layers and models via subclassing](https://www.tensorflow.org/guide/keras/making_new_layers_and_models_via_subclassing))
2. sequential API
3. functional API

---

---

**Approach-1**: here we build the model by subclassing the Keras $\texttt{Model}$ class followed by definition of of layers in $\texttt{__init__}$ and implementation of the model's forward pass in $\texttt{call()}$.

---

In [5]:
## Define 1-layer (softmax) neural network architecture
# Define model
class Softmax_Model(tf.keras.models.Model):
    def __init__(self):
        super(Softmax_Model, self).__init__()
        initializer = tf.keras.initializers.RandomUniform(minval=-0.5, maxval=0.5)
        self.dense1 = tf.keras.layers.Dense(num_labels, dtype = 'float64',\
                                 bias_initializer = initializer,\
                                 activation = tf.keras.activations.softmax)

    # Forward pass for the model
    def call(self, inputs):
        a = self.dense1(inputs)
        return a

In [36]:
## Define 2-layer (softmax) neural network architecture
# Define model
class model1(tf.keras.models.Model):
    def __init__(self):
        super(Softmax_Model, self).__init__()
        initializer = tf.keras.initializers.RandomUniform(minval=-0.5, maxval=0.5)
        self.layer1 = tf.keras.layers.Dense(num_labels, dtype = 'float64',\
                                 bias_initializer = initializer,\
                                 activation = tf.nn.leaky_relu)
        self.layer2 = tf.keras.layers.Dense(num_labels, dtype = 'float64',\
                                 bias_initializer = initializer,\
                                 activation = tf.nn.leaky_relu)
        

    # Forward pass for the model
    def call(self, inputs):
        a = self.layer1(inputs)
        a2 = self.layer2(a)
        return a

---

Build model

---

In [37]:
## Build softmax model
model = Softmax_Model()
batch_size = 100 # batch size
model.build((batch_size, num_features))

c:\Users\ATISHAY SG\anaconda3\envs\AIMLSem1\lib\site-packages\keras\src\initializers\initializers.py:120: UserWarning: The initializer RandomUniform is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


---

Compile and train the model on the training batches and test on the test set in one shot

---

In [31]:
## Compile model
opt = tf.keras.optimizers.Adam(learning_rate = 1e-03) # optimizer
loss_fn = tf.keras.losses.CategoricalCrossentropy()  # loss function
model.compile(optimizer = opt, loss = loss_fn, metrics = ['acc'])

# Train model and simultabeously test on the test set
model.fit(X_train, Y_train, batch_size = 100,\
          epochs = 10,\
          validation_data = (X_test, Y_test))

Epoch 1/10
600/600 [==============================] - 3s 4ms/step - loss: 0.6196 - acc: 0.8473 - val_loss: 0.3619 - val_acc: 0.9062
Epoch 2/10
600/600 [==============================] - 2s 3ms/step - loss: 0.3458 - acc: 0.9055 - val_loss: 0.3089 - val_acc: 0.9144
Epoch 3/10
600/600 [==============================] - 1s 2ms/step - loss: 0.3095 - acc: 0.9146 - val_loss: 0.2895 - val_acc: 0.9202
Epoch 4/10
600/600 [==============================] - 1s 2ms/step - loss: 0.2925 - acc: 0.9194 - val_loss: 0.2818 - val_acc: 0.9215
Epoch 5/10
600/600 [==============================] - 1s 2ms/step - loss: 0.2818 - acc: 0.9217 - val_loss: 0.2738 - val_acc: 0.9237
Epoch 6/10
600/600 [==============================] - 1s 2ms/step - loss: 0.2753 - acc: 0.9230 - val_loss: 0.2730 - val_acc: 0.9250
Epoch 7/10
600/600 [==============================] - 1s 2ms/step - loss: 0.2701 - acc: 0.9251 - val_loss: 0.2689 - val_acc: 0.9250
Epoch 8/10
600/600 [==============================] - 1s 2ms/step - loss: 0.

---

Instead of doing the above, we can explicitly write down the optimization step using $\texttt{GradientTape()}$ and train the model

---

In [34]:
## Create source dataset from input data (this is helpful for ppipelining later)
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
batch_size = 100 # batch size
# Create training batches
train_dataset = train_dataset.shuffle(buffer_size = 1024).batch(batch_size)

In [8]:
# Create softmax model
model = Softmax_Model()

opt = tf.keras.optimizers.Adam(learning_rate = 1e-03) # optimizer
loss_fn = tf.keras.losses.CategoricalCrossentropy()  # loss function

# Varible to store training loss per epoch
loss_train_epoch = tf.keras.metrics.Mean()

# Iterate over epochs
nepochs = 10
for epoch in range(nepochs):
  # Iterate over the batches of the dataset.
  for step, train_batch in enumerate(train_dataset):
    with tf.GradientTape() as g:
      # Compute loss
      yhat = model(train_batch[0])
      loss = loss_fn(train_batch[1], yhat)

    # Calculate gradients
    grad = g.gradient(loss, model.trainable_weights)

    # Update model
    opt.apply_gradients(zip(grad, model.trainable_weights))

    # Append training loss
    loss_train_epoch(loss)
  print('Epoch %d: train loss = %f'%(epoch+1, loss_train_epoch.result()))

Epoch 1: train loss = 0.634427
Epoch 2: train loss = 0.491448
Epoch 3: train loss = 0.431275
Epoch 4: train loss = 0.396825
Epoch 5: train loss = 0.374106
Epoch 6: train loss = 0.357738
Epoch 7: train loss = 0.345267
Epoch 8: train loss = 0.335411
Epoch 9: train loss = 0.327351
Epoch 10: train loss = 0.320637


In [9]:
# Compile model so it can be evaluated for test set
model.compile(optimizer = opt, loss = loss_fn, metrics = ['acc'])
print('\nAccuracy:', model.evaluate(X_test, Y_test, verbose=0)[1])


Accuracy: 0.9254999756813049


In [41]:
## Build 2-layer neural network model
model1 = Softmax_Model()
batch_size = 100 # batch size
model.build((batch_size, num_features))

In [43]:
## Compile model
opt = tf.keras.optimizers.Adam(learning_rate = 1e-03) # optimizer
loss_fn = tf.keras.losses.CategoricalCrossentropy()  # loss function
model.compile(optimizer = opt, loss = loss_fn, metrics = ['acc'])

# Train model and simultabeously test on the test set
model1.fit(X_train, Y_train, batch_size = 100,\
          epochs = 10,\
          validation_data = (X_test, Y_test))

RuntimeError: You must compile your model before training/testing. Use `model.compile(optimizer, loss)`.

In [35]:
# Create 2 layer model with 128 nodes in the hidden layer
model_2 = Softmax_Model()

opt = tf.keras.optimizers.Adam(learning_rate = 1e-03) # optimizer
loss_fn = tf.keras.losses.CategoricalCrossentropy()  # loss function

# Varible to store training loss per epoch
loss_train_epoch = tf.keras.metrics.Mean()

# Iterate over epochs
nepochs = 10
for epoch in range(nepochs):
  # Iterate over the batches of the dataset.
  for step, train_batch in enumerate(train_dataset):
    with tf.GradientTape() as g:
      # Compute loss
      yhat = model(train_batch[0])
      loss = loss_fn(train_batch[1], yhat)

    # Calculate gradients
    grad = g.gradient(loss, model.trainable_weights)

    # Update model
    opt.apply_gradients(zip(grad, model.trainable_weights))

    # Append training loss
    loss_train_epoch(loss)
  print('Epoch %d: train loss = %f'%(epoch+1, loss_train_epoch.result()))

Epoch 1: train loss = 0.637410
Epoch 2: train loss = 0.493021
Epoch 3: train loss = 0.432278
Epoch 4: train loss = 0.397571
Epoch 5: train loss = 0.374589
Epoch 6: train loss = 0.358094
Epoch 7: train loss = 0.345566
Epoch 8: train loss = 0.335634
Epoch 9: train loss = 0.327545
Epoch 10: train loss = 0.320802


---

**Approach-2**: here we build the model using the sequential API of TensorFlow Keras. Try this.

---

In [11]:
initializer = tf.keras.initializers.RandomUniform(minval=-0.5, maxval=0.5)
model_seq = tf.keras.models.Sequential([
    tf.keras.layers.Dense(num_labels, dtype = 'float64',\
                            bias_initializer = initializer,\
                            activation = tf.keras.activations.softmax),
    tf.keras.layers.Dense(1)
    ])
## Compile model
opt = tf.keras.optimizers.Adam(learning_rate = 1e-03) # optimizer
loss_fn = tf.keras.losses.CategoricalCrossentropy()  # loss function
model.compile(optimizer = opt, loss = loss_fn, metrics = ['acc'])

# Train model and simultabeously test on the test set
model.fit(X_train, Y_train, batch_size = 100,\
          epochs = 10,\
          validation_data = (X_test, Y_test))

Epoch 1/10
600/600 [==============================] - 2s 3ms/step - loss: 0.2567 - acc: 0.9288 - val_loss: 0.2630 - val_acc: 0.9272
Epoch 2/10
600/600 [==============================] - 1s 2ms/step - loss: 0.2541 - acc: 0.9291 - val_loss: 0.2630 - val_acc: 0.9261
Epoch 3/10
600/600 [==============================] - 1s 2ms/step - loss: 0.2527 - acc: 0.9300 - val_loss: 0.2637 - val_acc: 0.9271
Epoch 4/10
600/600 [==============================] - 1s 2ms/step - loss: 0.2509 - acc: 0.9302 - val_loss: 0.2643 - val_acc: 0.9268
Epoch 5/10
600/600 [==============================] - 1s 2ms/step - loss: 0.2494 - acc: 0.9313 - val_loss: 0.2644 - val_acc: 0.9275
Epoch 6/10
600/600 [==============================] - 1s 2ms/step - loss: 0.2482 - acc: 0.9309 - val_loss: 0.2620 - val_acc: 0.9280
Epoch 7/10
600/600 [==============================] - 1s 2ms/step - loss: 0.2468 - acc: 0.9322 - val_loss: 0.2620 - val_acc: 0.9281
Epoch 8/10
600/600 [==============================] - 1s 2ms/step - loss: 0.

In [13]:
from tensorflow.keras.optimizers import RMSprop
model_seq.compile(optimizer=RMSprop(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics = ['accuracy'])

---

**Approach-3**: here we build the model using the functional API of TensorFlow Keras. Try this.

---